In [10]:
import asyncio
import json
import websockets

from pynq import MMIO

HOST = "0.0.0.0"
PORT = 8765


# --------------------------------------------------
# AXI GPIO addresses
# --------------------------------------------------

GPIO_OUT_BASE = 0x41200000   # d + selector -> FPGA
GPIO_IN_BASE  = 0x41210000   # y0..y3 <- FPGA
GPIO_PHYS_BASE = 0x41220000  # BTN0 + SW0 + SW1

ADDRESS_RANGE = 0x1000

DATA_OFFSET = 0x00
TRI_OFFSET  = 0x04


gpio_out = MMIO(GPIO_OUT_BASE, ADDRESS_RANGE)
gpio_in = MMIO(GPIO_IN_BASE, ADDRESS_RANGE)
gpio_phys = MMIO(GPIO_PHYS_BASE, ADDRESS_RANGE)


# GPIO 0 = outputs toward FPGA
gpio_out.write(TRI_OFFSET, 0x0)

# GPIO 1 = inputs from FPGA
gpio_in.write(TRI_OFFSET, 0xF)

# GPIO 2 = physical inputs
gpio_phys.write(TRI_OFFSET, 0x7)


# --------------------------------------------------
# Connected WebSocket clients
# --------------------------------------------------

connected_clients = set()


# --------------------------------------------------
# MUX - still simulated in Python
# --------------------------------------------------

def mux(d0, d1, d2, d3, selector):
    inputs = [d0, d1, d2, d3]
    return inputs[selector]


# --------------------------------------------------
# DEMUX - REAL FPGA
# --------------------------------------------------

def demux_fpga(d, selector):
    d = int(bool(d))
    selector = int(selector)

    if selector < 0 or selector > 3:
        raise ValueError("selector must be between 0 and 3")

    # GPIO0 mapping:
    #
    # bit 0 = d
    # bit 1 = s0
    # bit 2 = s1
    #
    # selector 2 (10) + d=1
    # -> 0b101

    gpio_value = (selector << 1) | d

    gpio_out.write(DATA_OFFSET, gpio_value)

    # Read y0..y3 from AXI GPIO 1
    raw_outputs = gpio_in.read(DATA_OFFSET) & 0xF

    outputs = [
        (raw_outputs >> 0) & 1,  # y0
        (raw_outputs >> 1) & 1,  # y1
        (raw_outputs >> 2) & 1,  # y2
        (raw_outputs >> 3) & 1,  # y3
    ]

    return outputs


# --------------------------------------------------
# Physical inputs
# --------------------------------------------------

def read_physical_inputs():
    raw = gpio_phys.read(DATA_OFFSET) & 0x7

    d = (raw >> 0) & 1
    s0 = (raw >> 1) & 1
    s1 = (raw >> 2) & 1

    selector = (s1 << 1) | s0

    return {
        "raw": raw,
        "d": d,
        "s0": s0,
        "s1": s1,
        "selector": selector,
    }


async def broadcast(message):
    if not connected_clients:
        return

    dead_clients = []

    payload = json.dumps(message)

    for client in list(connected_clients):
        try:
            await client.send(payload)
        except websockets.ConnectionClosed:
            dead_clients.append(client)

    for client in dead_clients:
        connected_clients.discard(client)


async def monitor_physical_inputs():
    last_raw = None

    while True:
        physical = read_physical_inputs()

        if physical["raw"] != last_raw:
            # Also apply the physical state to the real DEMUX
            outputs = demux_fpga(
                physical["d"],
                physical["selector"],
            )

            message = {
                "type": "physical_inputs",
                "d": physical["d"],
                "s0": physical["s0"],
                "s1": physical["s1"],
                "selector": physical["selector"],
                "outputs": outputs,
            }

            print(
                f"Physical input changed: "
                f"D={physical['d']} "
                f"S1S0={physical['s1']}{physical['s0']} "
                f"outputs={outputs}"
            )

            await broadcast(message)

            last_raw = physical["raw"]

        await asyncio.sleep(0.05)


# --------------------------------------------------
# WebSocket
# --------------------------------------------------

async def handle_client(websocket):
    connected_clients.add(websocket)

    print(
        f"Client connected "
        f"({len(connected_clients)} client(s))"
    )

    try:
        async for message in websocket:
            data = json.loads(message)

            try:
                if data["type"] == "mux":

                    y = mux(
                        data["d0"],
                        data["d1"],
                        data["d2"],
                        data["d3"],
                        data["selector"],
                    )

                    response = {
                        "type": "mux_state",
                        "d0": data["d0"],
                        "d1": data["d1"],
                        "d2": data["d2"],
                        "d3": data["d3"],
                        "selector": data["selector"],
                        "y": y,
                    }

                elif data["type"] == "demux":

                    outputs = demux_fpga(
                        data["d"],
                        data["selector"],
                    )

                    response = {
                        "type": "demux_state",
                        "d": data["d"],
                        "selector": data["selector"],
                        "outputs": outputs,
                    }

                elif data["type"] == "get_physical_inputs":

                    physical = read_physical_inputs()

                    outputs = demux_fpga(
                        physical["d"],
                        physical["selector"],
                    )

                    response = {
                        "type": "physical_inputs",
                        "d": physical["d"],
                        "s0": physical["s0"],
                        "s1": physical["s1"],
                        "selector": physical["selector"],
                        "outputs": outputs,
                    }

                else:
                    response = {
                        "type": "error",
                        "message": "Unknown command",
                    }

            except Exception as error:
                response = {
                    "type": "error",
                    "message": str(error),
                }

            await websocket.send(json.dumps(response))

    except websockets.ConnectionClosed:
        print("Client disconnected")

    finally:
        connected_clients.discard(websocket)

        print(
            f"Client removed "
            f"({len(connected_clients)} client(s))"
        )


# --------------------------------------------------
# Start server + physical input monitor
# --------------------------------------------------

server = await websockets.serve(
    handle_client,
    HOST,
    PORT,
)

monitor_task = asyncio.create_task(
    monitor_physical_inputs()
)

print(f"WebSocket server running on ws://{HOST}:{PORT}")
print("Physical input monitoring started")

WebSocket server running on ws://0.0.0.0:8765
Physical input monitoring started
Physical input changed: D=0 S1S0=11 outputs=[0, 0, 0, 0]


Physical input changed: D=0 S1S0=00 outputs=[0, 0, 0, 0]
Client connected (1 client(s))
Physical input changed: D=0 S1S0=01 outputs=[0, 0, 0, 0]
Physical input changed: D=1 S1S0=01 outputs=[0, 1, 0, 0]
Physical input changed: D=0 S1S0=01 outputs=[0, 0, 0, 0]
Physical input changed: D=0 S1S0=00 outputs=[0, 0, 0, 0]
Physical input changed: D=0 S1S0=01 outputs=[0, 0, 0, 0]
Physical input changed: D=1 S1S0=01 outputs=[0, 1, 0, 0]
Physical input changed: D=0 S1S0=01 outputs=[0, 0, 0, 0]
Physical input changed: D=1 S1S0=01 outputs=[0, 1, 0, 0]
Physical input changed: D=0 S1S0=01 outputs=[0, 0, 0, 0]
Physical input changed: D=1 S1S0=01 outputs=[0, 1, 0, 0]
Physical input changed: D=0 S1S0=01 outputs=[0, 0, 0, 0]
Physical input changed: D=1 S1S0=01 outputs=[0, 1, 0, 0]
Physical input changed: D=0 S1S0=01 outputs=[0, 0, 0, 0]
Physical input changed: D=1 S1S0=01 outputs=[0, 1, 0, 0]
Physical input changed: D=0 S1S0=01 outputs=[0, 0, 0, 0]
Physical input changed: D=1 S1S0=01 outputs=[0, 1, 0, 0]


In [ ]:
from pynq import MMIO

print("import OK")

gpio_out = MMIO(0x41200000, 0x1000)
print("GPIO0 mmap OK")

gpio_out.write(0x04, 0x0)
print("GPIO0 direction OK")

gpio_out.write(0x00, 0b001)
print("GPIO0 write OK")

gpio_in = MMIO(0x41210000, 0x1000)
print("GPIO1 mmap OK")

gpio_in.write(0x04, 0xF)
print("GPIO1 direction OK")

value = gpio_in.read(0x00)
print(hex(value))

In [ ]:
for selector in range(4):
    value = (selector << 1) | 1

    gpio_out.write(0x00, value)

    written = gpio_out.read(0x00) & 0x7
    result = gpio_in.read(0x00) & 0xF

    print(
        f"selector={selector} "
        f"written={written:03b} "
        f"read={result:04b}"
    )

In [ ]:
async def test_demux():
    async with websockets.connect("ws://127.0.0.1:8765") as ws:
        for selector in range(4):
            await ws.send(json.dumps({
                "type": "demux",
                "d": 1,
                "selector": selector,
            }))

            response = json.loads(await ws.recv())
            print(response)

await test_demux()

In [ ]:
from pynq import MMIO

GPIO2_BASE = 0x41220000   # remplace si Vivado donne une autre adresse
gpio_phys = MMIO(GPIO2_BASE, 0x1000)

raw = gpio_phys.read(0x00) & 0x7

d  = (raw >> 0) & 1
s0 = (raw >> 1) & 1
s1 = (raw >> 2) & 1

print(f"raw={raw:03b}  D={d}  S0={s0}  S1={s1}")